# Lab 05 solution, notebook 1: a question that reaches level 3, and a skill of your own

The worked answer to the challenge of `05a_meet_context_and_skills.ipynb`. The setup repeats the notebook's skill
functions, tools and loop, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI(timeout=60, max_retries=2)

The house-sitter's skills, read from `house_skills/` as in Part E, and the `ask` loop from Part C. `read_skills`
does for any folder what Part E did for one: it reads every `SKILL.md` under it into a dictionary.

In [3]:
import yaml

LAB = ROOT / "05_Context_And_Skills"
BASE = (
    "You answer a house-sitter's questions for the owner of the flat, who is away for the weekend. "
    "Answer in one or two sentences."
)


def read_skills(folder):
    # Every SKILL.md under a folder: its description, its body and where it lives
    skills = {}
    for path in sorted(folder.glob("*/SKILL.md")):
        before, frontmatter, body = path.read_text().split("---", 2)
        meta = yaml.safe_load(frontmatter)
        skills[meta["name"]] = {"description": meta["description"], "body": body.strip(), "folder": path.parent}
    return skills


SKILLS = read_skills(LAB / "house_skills")
print(list(SKILLS))

['cat', 'plants', 'wifi']


In [4]:
def load_skill(name):
    # Level 2: the body of one SKILL.md
    return SKILLS[name]["body"]


def read_skill_file(skill, path):
    # Level 3: one file inside the skill's folder, and nothing outside it
    folder = SKILLS[skill]["folder"].resolve()
    target = (folder / path).resolve()
    if folder not in target.parents or not target.is_file():
        return "No such file in this skill."
    return target.read_text()

In [5]:
def ask(question, system, tools, functions, max_turns=8):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, reasoning_effort=EFFORT or omit
        )
        reply = response.choices[0].message
        print(f"call {turn + 1}: {response.usage.prompt_tokens} input tokens")

        # No tool call in the reply: it is the answer
        if not reply.tool_calls:
            return reply.content

        # Otherwise run each tool the model asked for and add the result to the conversation
        messages.append(reply)
        for call in reply.tool_calls:
            arguments = json.loads(call.function.arguments)
            print("   the model asked for:", call.function.name, arguments)
            result = functions[call.function.name](**arguments)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "Stopped: the model was still asking for tools after max_turns calls."

The tool schemas and the system prompt depend on which skills exist, so we build them in a function. Task 2 calls
it again after adding a skill.

In [6]:
def build(skills):
    # The two tool schemas and the system prompt for a set of skills
    tools = [
        {
            "type": "function",
            "function": {
                "name": "load_skill",
                "description": "Load the full notes of one skill.",
                "parameters": {
                    "type": "object",
                    "properties": {"name": {"type": "string", "enum": list(skills)}},
                    "required": ["name"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
        },
        {
            "type": "function",
            "function": {
                "name": "read_skill_file",
                "description": "Read a file inside a skill's folder, when the skill's notes point to it.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "skill": {"type": "string", "enum": list(skills)},
                        "path": {"type": "string", "description": "Path inside the skill folder"},
                    },
                    "required": ["skill", "path"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
        },
    ]
    skill_list = "\n".join(f"- {name}: {skill['description']}" for name, skill in skills.items())
    system = (
        BASE
        + " Answer only from the owner's skills. Each skill below holds the notes for one subject. "
        + "Load a skill with load_skill when a question needs it, and read a file inside it with read_skill_file "
        + "when the skill tells you to. If no skill covers a question, say so.\n\n"
        + "Skills:\n"
        + skill_list
    )
    return tools, system


skill_tools, WITH_SKILLS = build(SKILLS)
skill_functions = {"load_skill": load_skill, "read_skill_file": read_skill_file}

## Task 1: give the model a reason to go to level 3

The question describes a cat that seems ill. Our prediction before running: `load_skill` with `cat` first, because
the description says "or when she seems ill"; then `read_skill_file` with `references/when-she-is-ill.md`, because
the body's last lines say to read that file in this case; then the answer.

In [7]:
ILL = "Mishmish has not eaten since yesterday and she is hiding under the bed. What do I do?"

print(ask(ILL, WITH_SKILLS, skill_tools, skill_functions))

call 1: 410 input tokens
   the model asked for: load_skill {'name': 'cat'}


call 2: 631 input tokens
   the model asked for: read_skill_file {'skill': 'cat', 'path': 'references/when-she-is-ill.md'}


call 3: 829 input tokens
Check that her blue bowl has fresh water, the food is dry, and she isn’t shut in somewhere by mistake. Since she hasn’t eaten for a whole day and is hiding, call Dr Haddad at 06-555-0142 now, don’t give her medicine or milk, and text me; the vet is open 8 a.m.–8 p.m., and the same number reaches the night clinic after hours.


Three calls, as predicted:

1. `load_skill` with `cat` (410 input tokens).
2. `read_skill_file` with `references/when-she-is-ill.md` (631).
3. The answer (829): check the water, the food and the rooms, phone Dr Haddad on 06-555-0142, no medicine and no milk, text the owner.

Most of that answer (the vet, the number, no medicine, text the owner) is in the reference file and not in the body.
This is level 3, and the model reached it by following one sentence at the end of the body.

## Task 2: write a skill of your own

Our `bins` skill is in `solved/my_house_skills/bins/SKILL.md`. Yours goes in `my_house_skills/bins/` beside
`05a_meet_context_and_skills.ipynb`, and is added the same way. Its `name` is the folder's name, and its
`description` says what it holds and when to use it.

In [8]:
MY_SKILLS = LAB / "solved" / "my_house_skills"

print((MY_SKILLS / "bins" / "SKILL.md").read_text())

---
name: bins
description: Which bin goes out on which day, and where the bins stand. Use when the sitter asks about rubbish, recycling, bin days or where to put a full bag.
---

# Bins

- **General rubbish (grey bin):** goes out on Monday evening and is collected on Tuesday morning.
- **Recycling (blue bin):** goes out on Thursday evening, every week.
- **Glass:** does not go in a bin. The glass container is at the corner of the street, next to the bakery.
- The bins stand in the yard behind the building. Put them on the pavement by the front gate, and bring them back
  the next morning.



We add it to the three skills and build the tools and the prompt again. `SKILLS.update(...)` adds the new entry to
the dictionary that `load_skill` reads.

In [9]:
SKILLS.update(read_skills(MY_SKILLS))
skill_tools, WITH_SKILLS = build(SKILLS)

print(list(SKILLS))

['cat', 'plants', 'wifi', 'bins']


In [10]:
print(ask("What day do the bins go out?", WITH_SKILLS, skill_tools, skill_functions))

call 1: 441 input tokens
   the model asked for: load_skill {'name': 'bins'}


call 2: 570 input tokens
The grey bin goes out Monday evening for Tuesday collection, and the blue recycling bin goes out Thursday evening.


In [11]:
print(ask("How much do I feed Mishmish?", WITH_SKILLS, skill_tools, skill_functions))

call 1: 442 input tokens
   the model asked for: load_skill {'name': 'cat'}


call 2: 663 input tokens
Give Mishmish half a cup of dry food at 7 a.m. and half a cup at 6 p.m. (one cup total per day).


The bins question loaded `bins` and nothing else, and the answer has the two days from our file: the grey bin on Monday evening, the recycling on Thursday evening. The feeding question still loaded only `cat`.

**What the new skill costs a question that does not need it:** one more line in the list. The feeding question's first call sent 442 input tokens here, against 397 in the notebook's Part E run with three skills.